In [3]:
from nb_paths import PIPELINE_DIR, KWF_TEST_DIR
import sys, time
from pathlib import Path
import pandas as pd

HOME = Path.home()
CLIPS_DIR    = KWF_TEST_DIR / "clips"
OUTPUTS_DIR  = KWF_TEST_DIR / "outputs"
sys.path.insert(0, str(PIPELINE_DIR / "scripts"))

from extract_acoustic_features import process_split, FEATURE_COLS
print("Expected columns:", FEATURE_COLS)

Expected columns: ['clip_name', 'RMS_Energy', 'Spectral_Contrast', 'Spectral_Flatness', 'Spectral_Bandwidth', 'Spectral_Rolloff_85', 'Onset_Strength', 'MFCC_8', 'MFCC_9', 'MFCC_12', 'MFCC_13']


In [4]:
acoustic_csv = OUTPUTS_DIR / "nb04_acoustic_features.csv"

t0 = time.time()
process_split(
    split_name="Inference",
    split_dir=str(CLIPS_DIR),
    output_csv=str(acoustic_csv),
)
print(f"\nRuntime: {time.time() - t0:.1f}s")


Inference set: 1,200 files
Output: /Users/huyguy/kwf_test/outputs/nb04_acoustic_features.csv


Inference: 100%|██████████| 1200/1200 [00:36<00:00, 33.27it/s]


Done. 1,200 rows saved, 0 failed.

Runtime: 36.7s


In [5]:
df4 = pd.read_csv(acoustic_csv)

print("Shape:", df4.shape)
print("Columns match FEATURE_COLS:", df4.columns.tolist() == FEATURE_COLS)
print("Unique clip_names:", df4["clip_name"].nunique())
print("Example clip_name:", df4["clip_name"].iloc[0])
print("\nNulls per column:\n", df4.isna().sum())
print("\nInf values:", (df4.drop(columns="clip_name").abs() == float("inf")).sum().sum())

print("\nSummary:")
print(df4.drop(columns="clip_name").describe().T[["min", "mean", "max", "std"]])

# Key check against Step 2 output (does the clip_name format match?)
cnn = pd.read_csv(OUTPUTS_DIR / "nb02_tinycnn_predictions.csv")
exact = set(df4["clip_name"]) & set(cnn["clip_name"])
print("\nExact clip_name matches with Step 2:", len(exact))
stripped = set(cnn["clip_name"].str.replace(".wav", "", regex=False))
print("Matches after removing '.wav' from Step 2 names:", len(set(df4["clip_name"]) & stripped))

# Are features constant (useless) or wildly scaled?
print("\nZero-variance features:", df4.drop(columns="clip_name").columns[df4.drop(columns="clip_name").std() == 0].tolist())

Shape: (1200, 11)
Columns match FEATURE_COLS: True
Unique clip_names: 1200
Example clip_name: Audio_Moth_1_20250317_093124

Nulls per column:
 clip_name              0
RMS_Energy             0
Spectral_Contrast      0
Spectral_Flatness      0
Spectral_Bandwidth     0
Spectral_Rolloff_85    0
Onset_Strength         0
MFCC_8                 0
MFCC_9                 0
MFCC_12                0
MFCC_13                0
dtype: int64

Inf values: 0

Summary:
                             min         mean          max         std
RMS_Energy              0.002494     0.011137     0.301187    0.016115
Spectral_Contrast      17.229472    19.199003    24.002495    0.929059
Spectral_Flatness       0.009961     0.121169     0.341810    0.060805
Spectral_Bandwidth   2142.026028  2865.078883  3570.851819  257.512214
Spectral_Rolloff_85  3593.725962  8743.040125  9874.630033  672.681955
Onset_Strength          0.868508     1.039393     2.686687    0.206890
MFCC_8                 -9.187581    10.953791  

In [7]:
from nb_paths import PIPELINE_DIR, KWF_TEST_DIR
SPLIT_OUT = KWF_TEST_DIR / "split_clips"
acoustic5_csv = OUTPUTS_DIR / "nb04_recorder5_acoustic_features.csv"

t0 = time.time()
process_split(
    split_name="Inference",
    split_dir=str(SPLIT_OUT),
    output_csv=str(acoustic5_csv),
)
print(f"\nRuntime: {time.time() - t0:.1f}s")

a5 = pd.read_csv(acoustic5_csv)
print("Shape:", a5.shape, "| columns match:", a5.columns.tolist() == FEATURE_COLS)
print("Unique clip_names:", a5["clip_name"].nunique())
print("Nulls:", int(a5.isna().sum().sum()),
      "| Inf:", int((a5.drop(columns="clip_name").abs() == float("inf")).sum().sum()))

# Key check against Step 1 files, Step 2 and Step 3 outputs
on_disk = {p.name for p in SPLIT_OUT.glob("*.wav")}
print("\nExact match with Step 1 files:", len(set(a5["clip_name"]) & on_disk))
print("Match after adding '.wav':", len({n + '.wav' for n in a5['clip_name']} & on_disk))
b = pd.read_csv(OUTPUTS_DIR / "nb03_birdnet_output.csv")
print("Exact match with Step 3 output:", len(set(a5["clip_name"]) & set(b["clip_name"])))

print("\nSummary:")
print(a5.drop(columns="clip_name").describe().T[["min", "mean", "max", "std"]])


Inference set: 1,747 files
Output: /Users/huyguy/kwf_test/outputs/nb04_recorder5_acoustic_features.csv


Inference: 100%|██████████| 1747/1747 [00:42<00:00, 41.41it/s]


Done. 1,747 rows saved, 0 failed.

Runtime: 42.8s
Shape: (1747, 11) | columns match: True
Unique clip_names: 1747
Nulls: 0 | Inf: 0

Exact match with Step 1 files: 0
Match after adding '.wav': 1747
Exact match with Step 3 output: 0

Summary:
                             min         mean          max         std
RMS_Energy              0.022220     0.069477     0.359981    0.055782
Spectral_Contrast      17.304050    20.337112    21.863915    0.800990
Spectral_Flatness       0.012367     0.030621     0.131798    0.014456
Spectral_Bandwidth   1661.651966  2194.152123  3298.511552  439.834556
Spectral_Rolloff_85  6271.379770  8358.252122  8917.561974  313.413433
Onset_Strength          0.822927     0.984397     2.823488    0.170291
MFCC_8                  8.812487    28.291956    37.361916    3.693425
MFCC_9                -33.038414   -24.444702    -3.859092    3.269498
MFCC_12                 4.036610    14.191736    23.268419    2.505921
MFCC_13               -14.193281    -4.180945   

## Observations (Step 4)
- process_split(split_name, split_dir, output_csv) matches the call in run_pipeline.py step4_acoustic. 
- Input: 1,200 clips (recorder 1). Output: 1,200 rows x 11 columns (clip_name + 10 features), 29.6 s, 0 failed. 
- Columns match FEATURE_COLS exactly: RMS_Energy, Spectral_Contrast, Spectral_Flatness, Spectral_Bandwidth, Spectral_Rolloff_85, Onset_Strength, MFCC_8, MFCC_9, MFCC_12, MFCC_13.
- No nulls, no infinite values, no zero-variance features. Ranges are plausible (e.g., Spectral_Flatness within 0 to 1; Spectral_Rolloff_85 below the 11,025 Hz Nyquist limit at 22,050 Hz).
- Issue: clip_name has no ".wav" (e.g., Audio_Moth_1_20250317_093124), while Steps 2 and 3 include it. Exact matches with Step 2: 0 of 1,200; after stripping ".wav": 1,200 of 1,200. Step 7 must normalize the names. To be checked in Step 7.
- By design, Step 4 runs on all clips (not only meaningful ones); the meaningful filter is applied in Step 7 via keep_clips, and Volume_Spike_15s needs every clip.
- Recorder 5 (1,747 clips from Step 1): 1,747 rows x 11 columns, 0 failed, no nulls or infinities, 42.8 s (41 clips/s), same speed as recorder 1. clip_name has no ".wav": 0 exact matches with Step 1 files and Step 3 output, 1,747 after adding ".wav" (confirms the mismatch on a second recorder).
- Feature ranges differ from recorder 1 (e.g., mean RMS_Energy 0.069 vs 0.011; mean Spectral_Flatness 0.031 vs 0.121), consistent with a different recorder and time of day.